Kode her skal køres med conda miljø DAKI-NLP-mini

In [1]:
import torch
import torch_directml

# Data

In [2]:
import pandas as pd
import numpy as np


df = pd.read_json("train.jsonl", lines=True)

df.head()

,id,text,model,label,genre
0,ea468d03-1973-5039-86b2-ff225bb92c4e,"Duke Ellington, a titan of jazz, revolutionize...",falcon3-10b-instruct,1,essays
1,0d05f269-6d67-521d-9b5d-cc18f482c6c1,I reflected on the shifting dynamics of media ...,o3-mini,1,essays
2,c2ec79f3-da80-58f8-bef0-3e0ea7ab072f,"In F. Scott Fitzgerald's ""The Great Gatsby,"" t...",gpt-4o,1,essays
3,4ad37c58-0bb7-536b-997d-cfccabd0d094,I still chuckle when I think about that time I...,deepseek-r1-distill-qwen-32b,1,essays
4,07747b0c-5051-5e0d-8096-b4d4ed8bd98e,"Yoga, originating in ancient India, encompasse...",gemini-2.0-flash,1,essays


In [3]:

from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

device = torch_directml.device()
model = model.to(device)


c:\Users\Daniel K\anaconda3\envs\DAKI-NLP-mini\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Dataset

In [4]:

from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "dev.jsonl"
    }
)

print(dataset)
print(dataset["train"][0])


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'model', 'genre'],
        num_rows: 23707
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'model', 'genre'],
        num_rows: 1795
    })
})
{'id': 'ea468d03-1973-5039-86b2-ff225bb92c4e', 'text': 'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who contributed to this artistic symphony was Ivie Anderson, whose scat singing mirrored the improvisational prowess of musicians like Nanton. In "Ring Dem Bells," Ellington ingeniously interweaves scat singing as a dialogue with saxophones, creating a dynamic call and response that underscores his vision of music as a fluid conversation between voices and instruments.\n\nEllington consistently emphasized the voice as an instrument of equal importance to traditional brass and woodwinds, orchestra

Tokenisering

In [5]:

text_column = "text"
label_column = "label"

# Find de to klasser i træningsdata
class_names = sorted(set(dataset["train"][label_column]))
assert len(class_names) == 2

label_to_id = {
    label: idx for idx, label in enumerate(class_names)
}

print("Label mapping:", label_to_id)

def preprocess(batch):
    encoded = tokenizer(
        batch[text_column],
        truncation=True,
        max_length=256
    )

    encoded["label"] = [
        label_to_id[label]
        for label in batch[label_column]
    ]

    return encoded

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)


Label mapping: {0: 0, 1: 1}


Finetuning

In [6]:

import numpy as np

from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(
            labels,
            predictions,
            average="macro"
        )
    }

training_args = TrainingArguments(
    output_dir="./bert_authorship",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=42
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


Train model 

In [8]:
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [ ]:

from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm

# DataLoaders
train_loader = DataLoader(
    tokenized_dataset["train"],
    batch_size=training_args.per_device_train_batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    tokenized_dataset["validation"],
    batch_size=training_args.per_device_eval_batch_size,
    collate_fn=data_collator
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_args.learning_rate,
    weight_decay=training_args.weight_decay,
    foreach=False
)

best_f1 = -1
save_dir = Path(training_args.output_dir)

# Training
for epoch in range(int(training_args.num_train_epochs)):
    model.train()
    total_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{int(training_args.num_train_epochs)}"
    )

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)

            all_logits.append(outputs.logits.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())

    metrics = compute_metrics((
        np.concatenate(all_logits),
        np.concatenate(all_labels)
    ))

    print(f"Epoch {epoch + 1}")
    print(f"Train Loss: {total_loss / len(train_loader):.4f}")
    print(f"Val Accuracy: {metrics['accuracy']:.4f}")
    print(f"Val F1: {metrics['f1']:.4f}")

    # Gem den bedste model
    if metrics["f1"] > best_f1:
        best_f1 = metrics["f1"]

        cpu_weights = {
            k: v.detach().cpu()
            for k, v in model.state_dict().items()
        }

        model.save_pretrained(
            str(save_dir),
            state_dict=cpu_weights
        )
        tokenizer.save_pretrained(str(save_dir))

        print("Best model saved!")


c:\Users\Daniel K\anaconda3\envs\DAKI-NLP-mini\Lib\site-packages\torch\optim\adamw.py:380: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  exp_avg.lerp_(grad, 1 - beta1)


KeyboardInterrupt: 

Evaluering:

In [ ]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

results = trainer.evaluate()
print(results)

output = trainer.predict(
    tokenized_dataset["validation"]
)

predictions = np.argmax(
    output.predictions,
    axis=-1
)

print(
    classification_report(
        output.label_ids,
        predictions,
        target_names=[str(x) for x in class_names],
        zero_division=0
    )
)

print(
    confusion_matrix(
        output.label_ids,
        predictions
    )
)
